### Gold: `03_gold.zone_stress` — S(z) = Harrell-Davis 0.9 quantile of each zone's 210 ZSI values
Q̂ = Σ Wᵢ·x₍ᵢ₎,  Wᵢ = I_{i/n}(a,b) − I_{(i−1)/n}(a,b),  a = (n+1)p,  b = (n+1)(1−p)

Computed on the driver (263 × 210 values) with numpy/scipy so results are identical on every rerun,
then cross-checked with a pure Spark SQL version.

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

In [0]:
dbutils.widgets.text("settings", "spec")
s = ZSI_SETTINGS[dbutils.widgets.get("settings")]
suffix = s["table_suffix"]

In [0]:
windows = to_pandas_safe(spark.table(tbl("03_gold", f"zsi_windows{suffix}")))
n = int(windows.groupby("LocationID").size().iloc[0])
weights = hd_weights(n, s["quantile"])

S = {int(zone_id): float(np.dot(weights, np.sort(g["ZSI"].to_numpy(dtype=float))))
     for zone_id, g in windows.groupby("LocationID")}

zones = spark.table(tbl("02_silver", "taxi_zone_lookup")).filter("is_physical").orderBy("location_id").collect()
rows = [(z.location_id, z.zone, z.borough, S[z.location_id]) for z in zones]

spark.createDataFrame(rows, "LocationID INT, zone STRING, borough STRING, S DOUBLE") \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("03_gold", f"zone_stress{suffix}"))
spark.createDataFrame([(i + 1, float(w)) for i, w in enumerate(weights)], "i INT, weight DOUBLE") \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("03_gold", f"hd_weights{suffix}"))
print(f"S computed for {len(rows)} zones from {n} windows each")

#### Cross-check in Spark SQL (should be ~1e-12)

In [0]:
display(spark.sql(f"""
WITH ranked AS (
  SELECT LocationID, ZSI, ROW_NUMBER() OVER (PARTITION BY LocationID ORDER BY ZSI) AS i
  FROM {tbl('03_gold', f'zsi_windows{suffix}')}
), hd AS (
  SELECT r.LocationID, SUM(w.weight * r.ZSI) AS S_sql
  FROM ranked r JOIN {tbl('03_gold', f'hd_weights{suffix}')} w ON w.i = r.i
  GROUP BY r.LocationID
)
SELECT MAX(ABS(hd.S_sql - z.S)) AS max_abs_diff, COUNT(*) AS zones
FROM hd JOIN {tbl('03_gold', f'zone_stress{suffix}')} z USING (LocationID)"""))

In [0]:
display(spark.sql(f"""
SELECT LocationID, zone, borough, ROUND(S, 4) AS S
FROM {tbl('03_gold', f'zone_stress{suffix}')}
ORDER BY ROUND(S, 4) DESC, LocationID LIMIT 10"""))